# 🔍 SHAP Governance Framework for ML Models
> **A comprehensive, production-ready framework for explainable AI using SHAP**

[![License: MIT](https://img.shields.io/badge/License-MIT-teal.svg)](https://opensource.org/licenses/MIT)
[![Python 3.8+](https://img.shields.io/badge/python-3.8+-blue.svg)](https://www.python.org/)
[![SHAP](https://img.shields.io/badge/shap-latest-orange.svg)](https://shap.readthedocs.io/)

---

## 📋 Framework Overview

This notebook implements a **SHAP-based interpretability governance framework** covering:

| Section | Description |
|---------|-------------|
| **0. Setup & Data Loading** | Install dependencies, load your dataset |
| **1. Explainer Selection** | Auto-detect and configure the right SHAP explainer |
| **2. Feature Attribution Analysis** | Local & global SHAP explanations |
| **3. Visualisation Suite** | Full library of SHAP plots |
| **4. Governance Checks** | Automated compliance & quality gates |
| **5. Bias & Fairness Audit** | Protected attribute analysis via SHAP |
| **6. Drift Monitoring** | SHAP-based feature drift detection |
| **7. Model Card Generator** | Auto-generate explainability documentation |
| **8. Production Checklist** | Deployment readiness assessment |

---

### Supported Model Types
- 🌲 **Tree models** — XGBoost, LightGBM, RandomForest, GradientBoosting (`TreeExplainer`)
- 📈 **Linear models** — LogisticRegression, Ridge, Lasso, LinearSVC (`LinearExplainer`)
- 🧠 **Neural networks** — PyTorch, TensorFlow/Keras (`DeepExplainer`)
- ⬛ **Model-agnostic** — Any scikit-learn compatible model (`KernelExplainer`)

---

**Author:** Your Name  
**Version:** 1.0.0  
**Last Updated:** 2025

---
## Section 0 — Setup & Data Loading
### 0.1 Install Dependencies

In [ ]:
# Install required packages (run once)
# Uncomment the lines below if packages are not already installed

# !pip install shap pandas numpy scikit-learn matplotlib seaborn xgboost lightgbm
# !pip install ipywidgets scipy joblib
# !pip install torch torchvision  # optional: for neural network support
# !pip install tensorflow          # optional: alternative NN framework

print("✅ Dependencies ready. Proceed to imports.")

### 0.2 Imports & Configuration

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
from IPython.display import display, Markdown, HTML
import json, os, re

# ── SHAP ──────────────────────────────────────────────────────────────────────
import shap
shap.initjs()

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from scipy.stats import ks_2samp

# ── Optional tree models (graceful fallback) ──────────────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("ℹ️  XGBoost not installed — skipping XGB examples")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("ℹ️  LightGBM not installed — skipping LGB examples")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8fafc',
    'axes.edgecolor': '#e2e8f0',
    'axes.labelcolor': '#334155',
    'xtick.color': '#64748b',
    'ytick.color': '#64748b',
    'text.color': '#1e293b',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'sans-serif'
})

print(f"✅ SHAP version: {shap.__version__}")
print(f"✅ NumPy: {np.__version__} | Pandas: {pd.__version__}")
print(f"✅ Imports complete — framework ready")

---
### 0.3 ⬇️  Load YOUR Data Here

> **Instructions:**  
> 1. Replace the data loading block below with your own dataset  
> 2. Set `TARGET_COLUMN` to your target variable name  
> 3. Set `TASK_TYPE` to `'classification'` or `'regression'`  
> 4. Optionally list `PROTECTED_ATTRIBUTES` for bias auditing  
> 5. Run the cell — the rest of the notebook adapts automatically

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    🔧  CONFIGURE YOUR DATA HERE                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Option A: Load from CSV / Excel ───────────────────────────────────────────
# df = pd.read_csv('your_dataset.csv')
# df = pd.read_excel('your_dataset.xlsx')

# ── Option B: Load from file path ─────────────────────────────────────────────
# DATA_PATH = '/path/to/your/data.csv'
# df = pd.read_csv(DATA_PATH)

# ── Option C: Load from a database / API  ─────────────────────────────────────
# import sqlalchemy
# engine = sqlalchemy.create_engine('your_connection_string')
# df = pd.read_sql('SELECT * FROM your_table', engine)

# ── ⬇️  DEMO FALLBACK — replace this block with your own loading code ─────────
# Using sklearn's California Housing as a stand-in until you supply data
from sklearn.datasets import fetch_california_housing, load_breast_cancer

TASK_TYPE = 'regression'           # ← 'classification' or 'regression'
TARGET_COLUMN = 'MedHouseVal'      # ← Your target column name
PROTECTED_ATTRIBUTES = []          # ← e.g. ['age', 'gender', 'race'] for bias audit
MODEL_NAME = 'my_shap_model'       # ← Name for model card & logs

if TASK_TYPE == 'regression':
    dataset = fetch_california_housing(as_frame=True)
else:
    dataset = load_breast_cancer(as_frame=True)
    TARGET_COLUMN = 'target'

df = dataset.frame
# ── END DEMO FALLBACK ──────────────────────────────────────────────────────────

# ── Validate & summarise ───────────────────────────────────────────────────────
assert TARGET_COLUMN in df.columns, f"❌ Target column '{TARGET_COLUMN}' not found in dataframe!"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Auto-encode categoricals
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    print(f"   ↳ Label-encoded: {col}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n📦 Dataset loaded successfully")
print(f"   Rows: {len(df):,} | Features: {X.shape[1]} | Task: {TASK_TYPE.upper()}")
print(f"   Train set: {len(X_train):,} rows | Test set: {len(X_test):,} rows")
print(f"   Target: '{TARGET_COLUMN}'")
if PROTECTED_ATTRIBUTES:
    print(f"   Protected attributes: {PROTECTED_ATTRIBUTES}")
display(df.head(3))

### 0.4 Exploratory Data Summary

In [ ]:
def data_quality_report(df, target):
    """Generate a concise data quality summary."""
    n_missing = df.isnull().sum()
    pct_missing = (n_missing / len(df) * 100).round(2)
    dtypes = df.dtypes
    
    report = pd.DataFrame({
        'dtype': dtypes,
        'missing_count': n_missing,
        'missing_pct': pct_missing,
        'unique_values': df.nunique(),
        'mean': df.select_dtypes(include='number').mean().round(4)
    })
    
    print("📊 Data Quality Report")
    print(f"   Shape: {df.shape} | Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    print(f"   Columns with missing values: {(n_missing > 0).sum()}")
    print(f"   Duplicate rows: {df.duplicated().sum()}")
    print()
    display(report)
    
    return report

quality_report = data_quality_report(df, TARGET_COLUMN)

---
## Section 1 — Explainer Selection & Model Training
### 1.1 Model Registry

Train or load models for each supported type. Swap in your pre-trained model below.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.pipeline import Pipeline
import joblib

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║        🔧  SWAP IN YOUR PRE-TRAINED MODEL (optional)                    ║
# ║        e.g. model = joblib.load('my_model.pkl')                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

models = {}

# ── 1. Tree model (RandomForest) ──────────────────────────────────────────────
if TASK_TYPE == 'classification':
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
else:
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

rf_model.fit(X_train, y_train)
models['RandomForest'] = rf_model
print("✅ RandomForest trained")

# ── 2. XGBoost (if available) ─────────────────────────────────────────────────
if XGB_AVAILABLE:
    if TASK_TYPE == 'classification':
        xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, verbosity=0, eval_metric='logloss')
    else:
        xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    models['XGBoost'] = xgb_model
    print("✅ XGBoost trained")

# ── 3. LightGBM (if available) ────────────────────────────────────────────────
if LGB_AVAILABLE:
    if TASK_TYPE == 'classification':
        lgb_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
    else:
        lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
    lgb_model.fit(X_train, y_train)
    models['LightGBM'] = lgb_model
    print("✅ LightGBM trained")

# ── 4. Linear model ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

if TASK_TYPE == 'classification':
    linear_model = LogisticRegression(max_iter=1000, random_state=42)
else:
    linear_model = Ridge(alpha=1.0)

linear_model.fit(X_train_scaled, y_train)
models['Linear'] = linear_model
print("✅ Linear model trained")

# ── 5. Neural network (sklearn MLP) ──────────────────────────────────────────
if TASK_TYPE == 'classification':
    nn_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
else:
    nn_model = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)

nn_model.fit(X_train_scaled, y_train)
models['NeuralNetwork'] = nn_model
print("✅ Neural Network trained")

print(f"\n📦 Model registry: {list(models.keys())}")

### 1.2 Auto-Select SHAP Explainer

The framework auto-detects the optimal SHAP explainer for each model type.

In [ ]:
TREE_MODEL_TYPES = tuple(filter(None, [
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingClassifier,
    xgb.XGBRegressor if XGB_AVAILABLE else None,
    xgb.XGBClassifier if XGB_AVAILABLE else None,
    lgb.LGBMRegressor if LGB_AVAILABLE else None,
    lgb.LGBMClassifier if LGB_AVAILABLE else None,
]))

LINEAR_MODEL_TYPES = (LogisticRegression, Ridge, Lasso)

def get_explainer(model, X_background, model_name='model'):
    """
    Auto-detect and return the optimal SHAP explainer for a given model.
    
    Returns
    -------
    explainer : shap.Explainer subclass
    explainer_type : str
    """
    if isinstance(model, TREE_MODEL_TYPES):
        explainer = shap.TreeExplainer(model)
        explainer_type = 'TreeExplainer'

    elif isinstance(model, LINEAR_MODEL_TYPES):
        masker = shap.maskers.Independent(X_background)
        explainer = shap.LinearExplainer(model, masker)
        explainer_type = 'LinearExplainer'

    elif hasattr(model, 'predict') and hasattr(model, 'coefs_'):
        # sklearn MLP — use KernelExplainer with a sampled background
        bg = shap.sample(X_background, min(100, len(X_background)))
        predict_fn = model.predict_proba if TASK_TYPE == 'classification' else model.predict
        explainer = shap.KernelExplainer(predict_fn, bg)
        explainer_type = 'KernelExplainer (MLP)'

    else:
        # Fallback: model-agnostic KernelExplainer
        bg = shap.sample(X_background, min(100, len(X_background)))
        predict_fn = model.predict_proba if (TASK_TYPE == 'classification' and hasattr(model, 'predict_proba')) else model.predict
        explainer = shap.KernelExplainer(predict_fn, bg)
        explainer_type = 'KernelExplainer (agnostic)'

    print(f"   {model_name:20s} → {explainer_type}")
    return explainer, explainer_type


print("🔍 Selecting SHAP explainers...")
explainers = {}
explainer_types = {}

for name, model in models.items():
    X_bg = X_train_scaled if name in ('Linear', 'NeuralNetwork') else X_train
    exp, exp_type = get_explainer(model, X_bg, name)
    explainers[name] = exp
    explainer_types[name] = exp_type

print("\n✅ Explainer selection complete")

---
## Section 2 — Feature Attribution Analysis
### 2.1 Compute SHAP Values

In [ ]:
# ── Configure sample size for SHAP computation ────────────────────────────────
# For KernelExplainer, use smaller samples to manage compute time.
# Increase SHAP_SAMPLE_SIZE for more accurate global estimates.
SHAP_SAMPLE_SIZE = 200   # ← adjust as needed

shap_values_store = {}

for name, exp in explainers.items():
    X_eval = X_test_scaled if name in ('Linear', 'NeuralNetwork') else X_test
    sample = X_eval.iloc[:SHAP_SAMPLE_SIZE]

    print(f"⏳ Computing SHAP values for {name}...", end=' ')
    sv = exp(sample)
    shap_values_store[name] = sv
    print("Done ✅")

# Primary model for detailed analysis (first tree model, or first available)
PRIMARY_MODEL = next(
    (n for n in ['XGBoost', 'LightGBM', 'RandomForest'] if n in shap_values_store),
    list(shap_values_store.keys())[0]
)
print(f"\n📌 Primary model for detailed analysis: {PRIMARY_MODEL}")

### 2.2 Global Feature Importance Comparison

In [ ]:
def get_mean_abs_shap(sv, feature_names):
    """Compute mean |SHAP| per feature, handling multi-output."""
    vals = sv.values
    if vals.ndim == 3:        # multi-class: (samples, features, classes)
        vals = np.abs(vals).mean(axis=2)
    return pd.Series(np.abs(vals).mean(axis=0), index=feature_names)


feature_names = X_test.columns.tolist()
importance_df = pd.DataFrame()

for name, sv in shap_values_store.items():
    imp = get_mean_abs_shap(sv, feature_names)
    importance_df[name] = imp

importance_df = importance_df.sort_values(PRIMARY_MODEL, ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
n_models = len(importance_df.columns)
fig, ax = plt.subplots(figsize=(11, max(5, len(importance_df) * 0.38)))

x = np.arange(len(importance_df))
width = 0.8 / n_models
palette = ['#0d9488', '#6366f1', '#f59e0b', '#ef4444', '#84cc16']

for i, col in enumerate(importance_df.columns):
    bars = ax.barh(x + i * width, importance_df[col].values, width * 0.9,
                   label=col, color=palette[i % len(palette)], alpha=0.88)

ax.set_yticks(x + width * (n_models - 1) / 2)
ax.set_yticklabels(importance_df.index, fontsize=10)
ax.set_xlabel('Mean |SHAP Value|', fontsize=11)
ax.set_title('Global Feature Importance — All Models (SHAP)', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='lower right', framealpha=0.9)
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('shap_global_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Top 5 features (by primary model):")
display(importance_df.head(5).round(4))

### 2.3 Local Explanation — Single Prediction

In [ ]:
# ── Inspect a specific prediction ─────────────────────────────────────────────
SAMPLE_IDX = 0   # ← Change this to inspect different predictions

sv_primary = shap_values_store[PRIMARY_MODEL]

print(f"🔎 Local explanation for sample index {SAMPLE_IDX} ({PRIMARY_MODEL})\n")

# Handle multi-class: show class 1 (positive class)
if sv_primary.values.ndim == 3:
    sv_display = sv_primary[SAMPLE_IDX, :, 1]
else:
    sv_display = sv_primary[SAMPLE_IDX]

shap.plots.waterfall(sv_display, max_display=15)

### 2.4 Attribution Stability Test

> **Governance standard:** SHAP values should be stable across repeated runs and similar inputs. High variance indicates model instability.

In [ ]:
def attribution_stability_test(explainer, X_sample, n_bootstrap=5):
    """
    Test SHAP value stability via bootstrap resampling.
    Returns coefficient of variation per feature.
    """
    results = []
    sample = X_sample.iloc[:50]  # Use small subset for speed

    for seed in range(n_bootstrap):
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(sample), size=len(sample), replace=True)
        sv = explainer(sample.iloc[idx])
        vals = sv.values
        if vals.ndim == 3:
            vals = vals[:, :, 1]
        results.append(np.abs(vals).mean(axis=0))

    results = np.array(results)           # (n_bootstrap, n_features)
    mean_imp = results.mean(axis=0)
    std_imp  = results.std(axis=0)
    cv       = np.where(mean_imp > 1e-8, std_imp / mean_imp, 0.0)  # Coefficient of variation

    stability_df = pd.DataFrame({
        'feature': X_sample.columns,
        'mean_shap': mean_imp.round(5),
        'std_shap':  std_imp.round(5),
        'cv':        cv.round(4)
    }).sort_values('cv', ascending=False)

    return stability_df


print(f"🧪 Running attribution stability test ({PRIMARY_MODEL})...")
X_eval_primary = X_test_scaled if PRIMARY_MODEL in ('Linear', 'NeuralNetwork') else X_test

stability_df = attribution_stability_test(explainers[PRIMARY_MODEL], X_eval_primary)

# Governance threshold
CV_THRESHOLD = 0.15
unstable = stability_df[stability_df['cv'] > CV_THRESHOLD]

print(f"\n{'⚠️ ' if len(unstable) else '✅ '}Attribution Stability Results (CV threshold: {CV_THRESHOLD})")
display(stability_df.head(10))

if len(unstable):
    print(f"\n⚠️  {len(unstable)} feature(s) exceed stability threshold (CV > {CV_THRESHOLD}):")
    print(f"   {unstable['feature'].tolist()}")
    print("   → Consider regularisation, more training data, or feature engineering.")
else:
    print("\n✅ All features pass stability threshold.")

---
## Section 3 — Visualisation Suite
### 3.1 Summary Beeswarm Plot

In [ ]:
sv = shap_values_store[PRIMARY_MODEL]

if sv.values.ndim == 3:
    # Multi-class: plot class 1
    import shap as _shap
    sv_plot = _shap.Explanation(
        values=sv.values[:, :, 1],
        base_values=sv.base_values[:, 1] if sv.base_values.ndim > 1 else sv.base_values,
        data=sv.data,
        feature_names=sv.feature_names
    )
else:
    sv_plot = sv

print(f"📊 SHAP Summary Plot — {PRIMARY_MODEL}")
shap.summary_plot(sv_plot, max_display=15, show=True)

### 3.2 Feature Dependence Plots (Top 3 Features)

In [ ]:
top_features = importance_df[PRIMARY_MODEL].nlargest(3).index.tolist()
sv_vals = sv_plot.values
data_vals = sv_plot.data if hasattr(sv_plot.data, 'values') else sv_plot.data
if hasattr(data_vals, 'values'):
    data_arr = data_vals.values
else:
    data_arr = np.array(data_vals)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
palette_dep = ['#0d9488', '#6366f1', '#f59e0b']

for i, feat in enumerate(top_features):
    feat_idx = feature_names.index(feat)
    ax = axes[i]
    ax.scatter(data_arr[:, feat_idx], sv_vals[:, feat_idx],
               alpha=0.5, s=18, c=palette_dep[i], edgecolors='none')
    ax.axhline(0, color='#94a3b8', linewidth=0.8, linestyle='--')
    ax.set_xlabel(feat, fontsize=11)
    ax.set_ylabel('SHAP Value', fontsize=11)
    ax.set_title(f'Dependence: {feat}', fontsize=12, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle(f'SHAP Dependence Plots — {PRIMARY_MODEL}', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('shap_dependence_plots.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 SHAP Decision Plot — Multiple Predictions

In [ ]:
N_DECISION = min(20, len(sv_plot))

base = sv_plot.base_values[0] if sv_plot.base_values.ndim > 0 else float(sv_plot.base_values)

print(f"📈 Decision Plot — {N_DECISION} predictions ({PRIMARY_MODEL})")
shap.decision_plot(
    base,
    sv_vals[:N_DECISION],
    feature_names=feature_names,
    show=True,
    ignore_warnings=True
)

### 3.4 SHAP Heatmap — Feature Attribution Across Samples

In [ ]:
TOP_N = min(10, len(feature_names))
top_idx = importance_df[PRIMARY_MODEL].nlargest(TOP_N).index.tolist()
top_col_idx = [feature_names.index(f) for f in top_idx]

heatmap_data = sv_vals[:, top_col_idx]

fig, ax = plt.subplots(figsize=(12, 6))
vmax = np.abs(heatmap_data).quantile(0.95) if hasattr(np.abs(heatmap_data), 'quantile') else np.percentile(np.abs(heatmap_data), 95)

im = ax.imshow(heatmap_data.T, aspect='auto', cmap='RdBu_r',
                vmin=-vmax, vmax=vmax, interpolation='nearest')

ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_idx, fontsize=10)
ax.set_xlabel('Sample Index', fontsize=11)
ax.set_title(f'SHAP Value Heatmap — Top {TOP_N} Features ({PRIMARY_MODEL})',
             fontsize=13, fontweight='bold', pad=10)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('SHAP Value', fontsize=10)
plt.tight_layout()
plt.savefig('shap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 4 — Governance Checks
### 4.1 Explainability Quality Gates

> These automated checks implement explainability standards for production deployment.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              🔧  GOVERNANCE THRESHOLDS — CONFIGURE HERE                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
GOVERNANCE_CONFIG = {
    'min_top3_coverage': 0.60,   # Top-3 features should explain ≥ 60% of total SHAP
    'max_single_feature_dominance': 0.70,  # No single feature should explain > 70%
    'max_cv_threshold': 0.15,    # Attribution stability: max coefficient of variation
    'min_feature_count': 3,      # Minimum number of features with non-trivial SHAP
    'trivial_shap_threshold': 0.01,  # Below this = trivial contribution
}

def run_governance_checks(shap_values_dict, feature_names, config):
    """
    Run all governance quality gates across all models.
    Returns a results dict with pass/fail per check per model.
    """
    results = {}

    for model_name, sv in shap_values_dict.items():
        vals = sv.values
        if vals.ndim == 3:
            vals = vals[:, :, 1]

        mean_abs = np.abs(vals).mean(axis=0)
        total_shap = mean_abs.sum()
        sorted_idx = np.argsort(mean_abs)[::-1]

        top3_coverage   = mean_abs[sorted_idx[:3]].sum() / (total_shap + 1e-10)
        top1_dominance  = mean_abs[sorted_idx[0]]  / (total_shap + 1e-10)
        nontrivial_feat = int((mean_abs / (total_shap + 1e-10) > config['trivial_shap_threshold']).sum())

        checks = {
            'top3_coverage': {
                'value': round(top3_coverage, 4),
                'threshold': config['min_top3_coverage'],
                'pass': top3_coverage >= config['min_top3_coverage'],
                'description': 'Top-3 features explain ≥ 60% of SHAP mass'
            },
            'single_feature_dominance': {
                'value': round(top1_dominance, 4),
                'threshold': config['max_single_feature_dominance'],
                'pass': top1_dominance <= config['max_single_feature_dominance'],
                'description': 'No single feature explains > 70% of SHAP mass'
            },
            'nontrivial_features': {
                'value': nontrivial_feat,
                'threshold': config['min_feature_count'],
                'pass': nontrivial_feat >= config['min_feature_count'],
                'description': f'≥ {config["min_feature_count"]} features contribute meaningfully'
            },
            'shap_sum_check': {
                'value': round(total_shap, 4),
                'threshold': None,
                'pass': total_shap > 0,
                'description': 'SHAP values are non-zero (explainer is functional)'
            }
        }
        results[model_name] = checks

    return results


gov_results = run_governance_checks(shap_values_store, feature_names, GOVERNANCE_CONFIG)

# ── Display results table ──────────────────────────────────────────────────────
rows = []
for model_name, checks in gov_results.items():
    for check_name, detail in checks.items():
        rows.append({
            'Model': model_name,
            'Check': check_name,
            'Description': detail['description'],
            'Value': detail['value'],
            'Threshold': detail['threshold'],
            'Status': '✅ PASS' if detail['pass'] else '❌ FAIL'
        })

gov_df = pd.DataFrame(rows)
print("🏛️  Governance Quality Gates")
display(gov_df.style.applymap(
    lambda v: 'background-color: #d1fae5' if v == '✅ PASS' else ('background-color: #fee2e2' if v == '❌ FAIL' else ''),
    subset=['Status']
))

total_checks = len(rows)
passed = gov_df['Status'].str.contains('PASS').sum()
print(f"\n📋 Result: {passed}/{total_checks} checks passed")

### 4.2 SHAP Additive Consistency Validation

> For TreeExplainer and LinearExplainer, SHAP values should sum to `prediction − base_value`. This cell validates that property.

In [ ]:
def validate_shap_additivity(model_name, model, explainer, sv, X_eval, task_type, tol=1e-2):
    """
    Validate SHAP additive consistency:
    sum(SHAP values) ≈ prediction − base_value
    """
    exp_type = explainer_types[model_name]
    
    # Only exact for TreeExplainer and LinearExplainer
    if 'Kernel' in exp_type:
        print(f"   {model_name}: Skipped (KernelExplainer gives approximate values)")
        return None

    vals = sv.values
    if vals.ndim == 3:
        vals = vals[:, :, 1]                     # binary class 1
    base = sv.base_values
    if base.ndim > 1:
        base = base[:, 1]

    shap_sum = vals.sum(axis=1) + base

    if task_type == 'classification' and hasattr(model, 'predict_proba'):
        preds = model.predict_proba(X_eval.iloc[:len(vals)])[:, 1]
    else:
        preds = model.predict(X_eval.iloc[:len(vals)])

    preds = preds[:len(shap_sum)]
    max_err = np.abs(shap_sum - preds).max()
    mean_err = np.abs(shap_sum - preds).mean()
    passed = max_err < tol

    status = '✅ PASS' if passed else '⚠️  WARN'
    print(f"   {model_name:20s} | max_err={max_err:.6f} | mean_err={mean_err:.6f} | {status}")
    return {'model': model_name, 'max_err': max_err, 'mean_err': mean_err, 'passed': passed}


print("🔬 SHAP Additive Consistency Validation (tolerance = 0.01)\n")
additive_results = []
for name, exp in explainers.items():
    X_eval = X_test_scaled if name in ('Linear', 'NeuralNetwork') else X_test
    res = validate_shap_additivity(name, models[name], exp, shap_values_store[name], X_eval, TASK_TYPE)
    if res:
        additive_results.append(res)

---
## Section 5 — Bias & Fairness Audit
### 5.1 Protected Attribute SHAP Contribution

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   🔧  Set PROTECTED_ATTRIBUTES at the top of the notebook               ║
# ║   e.g. PROTECTED_ATTRIBUTES = ['age', 'gender', 'race']                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

BIAS_THRESHOLD = 0.05    # Flag if protected attribute explains > 5% of total SHAP

def run_bias_audit(model_name, sv, feature_names, protected_attrs, threshold):
    vals = sv.values
    if vals.ndim == 3:
        vals = vals[:, :, 1]

    mean_abs = np.abs(vals).mean(axis=0)
    total = mean_abs.sum()

    audit_rows = []
    for attr in protected_attrs:
        if attr not in feature_names:
            print(f"   ⚠️  '{attr}' not found in feature names — skipping")
            continue
        idx = feature_names.index(attr)
        contribution = mean_abs[idx] / (total + 1e-10)
        flagged = contribution > threshold
        audit_rows.append({
            'Model': model_name,
            'Protected Attribute': attr,
            'Mean |SHAP|': round(mean_abs[idx], 5),
            '% of Total SHAP': f"{contribution*100:.2f}%",
            'Threshold': f"{threshold*100:.0f}%",
            'Status': '🚨 FLAGGED' if flagged else '✅ OK'
        })
    return audit_rows


if not PROTECTED_ATTRIBUTES:
    display(Markdown("""
> **ℹ️  No protected attributes configured.**  
> To enable the bias audit, set `PROTECTED_ATTRIBUTES = ['col1', 'col2', ...]`  
> at the top of the notebook (Section 0.3).
"""))
else:
    print(f"🔍 Bias Audit — Protected Attributes: {PROTECTED_ATTRIBUTES}")
    all_audit = []
    for name, sv in shap_values_store.items():
        all_audit.extend(run_bias_audit(name, sv, feature_names, PROTECTED_ATTRIBUTES, BIAS_THRESHOLD))

    if all_audit:
        audit_df = pd.DataFrame(all_audit)
        display(audit_df.style.applymap(
            lambda v: 'background-color: #fee2e2; font-weight: bold' if '🚨' in str(v) else
                      ('background-color: #d1fae5' if '✅' in str(v) else ''),
            subset=['Status']
        ))
        flagged_count = sum('FLAGGED' in r['Status'] for r in all_audit)
        print(f"\n📋 {flagged_count} protected attribute(s) flagged above threshold ({BIAS_THRESHOLD*100:.0f}%)")

### 5.2 Intersectional SHAP Distribution Analysis

In [ ]:
def shap_distribution_by_group(model_name, sv, X_eval, groupby_col, target_feature, n_bins=4):
    """
    Compare SHAP value distributions for a feature across demographic groups.
    Uses binning if groupby_col is continuous.
    """
    vals = sv.values
    if vals.ndim == 3:
        vals = vals[:, :, 1]

    target_idx = feature_names.index(target_feature)
    shap_col = vals[:, target_idx]

    df_plot = pd.DataFrame({'shap': shap_col}).reset_index(drop=True)
    group_vals = X_eval[groupby_col].values[:len(shap_col)]

    if X_eval[groupby_col].nunique() > n_bins:
        df_plot['group'] = pd.cut(group_vals, bins=n_bins, labels=[f'Q{i+1}' for i in range(n_bins)])
    else:
        df_plot['group'] = group_vals

    fig, ax = plt.subplots(figsize=(9, 4))
    groups = df_plot['group'].dropna().unique()
    palette = ['#0d9488', '#6366f1', '#f59e0b', '#ef4444']

    for i, grp in enumerate(sorted(groups, key=str)):
        data = df_plot[df_plot['group'] == grp]['shap']
        ax.hist(data, bins=20, alpha=0.55, label=str(grp), color=palette[i % len(palette)], edgecolor='none')

    ax.axvline(0, color='#94a3b8', linestyle='--', linewidth=1)
    ax.set_xlabel(f'SHAP Value for "{target_feature}"', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'SHAP Distribution by {groupby_col} — {model_name}', fontsize=12, fontweight='bold')
    ax.legend(title=groupby_col, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


if PROTECTED_ATTRIBUTES:
    groupby = PROTECTED_ATTRIBUTES[0]
    target_feat = importance_df[PRIMARY_MODEL].index[0]   # top feature
    X_eval_p = X_test_scaled if PRIMARY_MODEL in ('Linear', 'NeuralNetwork') else X_test
    if groupby in X_eval_p.columns:
        shap_distribution_by_group(PRIMARY_MODEL, shap_values_store[PRIMARY_MODEL], X_eval_p, groupby, target_feat)
    else:
        print(f"ℹ️  '{groupby}' not in evaluation data — skipping group distribution plot.")
else:
    # Demo with top 2 features as proxy groups
    top2 = importance_df[PRIMARY_MODEL].index[:2].tolist()
    X_eval_p = X_test_scaled if PRIMARY_MODEL in ('Linear', 'NeuralNetwork') else X_test
    print(f"ℹ️  No protected attributes set. Showing distribution example using '{top2[0]}' as groupby.")
    shap_distribution_by_group(PRIMARY_MODEL, shap_values_store[PRIMARY_MODEL], X_eval_p, top2[0], top2[1])

---
## Section 6 — SHAP-Based Drift Monitoring
### 6.1 Feature Attribution Drift Detection

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   In production: replace X_reference / X_current with real windows      ║
# ║   e.g. last month's data vs this month's data                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DRIFT_KS_THRESHOLD = 0.10   # KS statistic — drift flagged above this value

def compute_shap_drift(explainer, X_reference, X_current, feature_names, ks_threshold=0.10):
    """
    Detect SHAP attribution drift between a reference and current window.
    Uses Kolmogorov-Smirnov test per feature.
    """
    sv_ref = explainer(X_reference)
    sv_cur = explainer(X_current)

    def extract(sv):
        v = sv.values
        return v[:, :, 1] if v.ndim == 3 else v

    vals_ref = extract(sv_ref)
    vals_cur = extract(sv_cur)

    drift_rows = []
    for i, feat in enumerate(feature_names):
        stat, p_val = ks_2samp(vals_ref[:, i], vals_cur[:, i])
        drift_rows.append({
            'Feature': feat,
            'KS Statistic': round(stat, 4),
            'P-Value': round(p_val, 5),
            'Drifted': stat > ks_threshold
        })

    drift_df = pd.DataFrame(drift_rows).sort_values('KS Statistic', ascending=False)
    return drift_df


# Simulate reference vs current windows from test set
half = len(X_test) // 2
X_eval_primary = X_test_scaled if PRIMARY_MODEL in ('Linear', 'NeuralNetwork') else X_test

X_ref_window = X_eval_primary.iloc[:half]
X_cur_window = X_eval_primary.iloc[half:half * 2]

print("🔭 Computing SHAP drift (reference vs. current window)...")
drift_df = compute_shap_drift(explainers[PRIMARY_MODEL], X_ref_window, X_cur_window, feature_names, DRIFT_KS_THRESHOLD)

n_drifted = drift_df['Drifted'].sum()
print(f"\n{'⚠️' if n_drifted else '✅'} {n_drifted}/{len(drift_df)} features show SHAP drift (KS > {DRIFT_KS_THRESHOLD})")
display(drift_df.head(10).style.applymap(
    lambda v: 'background-color: #fee2e2' if v is True else ('background-color: #d1fae5' if v is False else ''),
    subset=['Drifted']
))

### 6.2 Drift Visualisation

In [ ]:
top_drift = drift_df.head(6)['Feature'].tolist()

sv_ref = explainers[PRIMARY_MODEL](X_ref_window)
sv_cur = explainers[PRIMARY_MODEL](X_cur_window)

def get_vals(sv):
    v = sv.values
    return v[:, :, 1] if v.ndim == 3 else v

ref_vals = get_vals(sv_ref)
cur_vals = get_vals(sv_cur)

ncols = 3
nrows = (len(top_drift) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(top_drift):
    feat_idx = feature_names.index(feat)
    ax = axes[i]
    ax.hist(ref_vals[:, feat_idx], bins=25, alpha=0.6, label='Reference', color='#0d9488', edgecolor='none')
    ax.hist(cur_vals[:, feat_idx], bins=25, alpha=0.6, label='Current', color='#f59e0b', edgecolor='none')

    ks = drift_df[drift_df['Feature'] == feat]['KS Statistic'].values[0]
    flagged = ks > DRIFT_KS_THRESHOLD
    title_color = '#ef4444' if flagged else '#1e293b'
    ax.set_title(f'{feat}\nKS = {ks:.4f} {"⚠️" if flagged else "✅"}',
                 fontsize=10, fontweight='bold', color=title_color)
    ax.set_xlabel('SHAP Value', fontsize=9)
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'SHAP Drift — {PRIMARY_MODEL}', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('shap_drift_monitor.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7 — Model Card Generator
### 7.1 Auto-Generate Explainability Model Card

In [ ]:
def generate_model_card(model_name, task_type, feature_names, shap_importance,
                        gov_results, drift_df, protected_attrs, explainer_type_map):
    """
    Generate a Markdown model card with embedded explainability information.
    """
    now = datetime.now().strftime('%Y-%m-%d %H:%M')
    top5 = shap_importance.nlargest(5)
    n_pass = sum(1 for c in gov_results.values() if c['pass'])
    n_total = len(gov_results)
    n_drifted = int(drift_df['Drifted'].sum())

    top5_rows = '\n'.join(
        f'| {i+1} | {feat} | {val:.4f} |'
        for i, (feat, val) in enumerate(top5.items())
    )

    gov_rows = '\n'.join(
        f"| {k} | {v['description']} | {v['value']} | {'✅ PASS' if v['pass'] else '❌ FAIL'} |"
        for k, v in gov_results.items()
    )

    protected_section = (
        f"Protected attributes monitored: {', '.join(f'`{a}`' for a in protected_attrs)}"
        if protected_attrs else
        "No protected attributes configured. Set `PROTECTED_ATTRIBUTES` for bias auditing."
    )

    card = f"""# Model Card: {model_name}

> Auto-generated by SHAP Governance Framework · {now}

---

## 1. Model Overview

| Field | Value |
|-------|-------|
| **Model Name** | {model_name} |
| **Task Type** | {task_type.capitalize()} |
| **SHAP Explainer** | {explainer_type_map.get(model_name, 'N/A')} |
| **Features** | {len(feature_names)} |
| **Generated** | {now} |

---

## 2. Top 5 Features by SHAP Importance

| Rank | Feature | Mean |SHAP| |
|------|---------|-------------|
{top5_rows}

---

## 3. Governance Check Results

**Overall: {n_pass}/{n_total} checks passed**

| Check | Description | Value | Status |
|-------|-------------|-------|--------|
{gov_rows}

---

## 4. Bias & Fairness

{protected_section}

---

## 5. Drift Monitoring

- **Features monitored:** {len(drift_df)}
- **Features with detected drift (KS > {DRIFT_KS_THRESHOLD}):** {n_drifted}
- **Top drifted features:** {', '.join(drift_df[drift_df['Drifted']].head(3)['Feature'].tolist()) or 'None'}

---

## 6. Explainability Standards Applied

- ✅ SHAP additive consistency validated  
- ✅ Attribution stability tested (bootstrap resampling)  
- ✅ Global & local explanations generated  
- ✅ Drift monitoring baseline established  
- {'✅' if protected_attrs else '⬜'} Bias audit on protected attributes  

---

## 7. Intended Use & Limitations

> **Fill in:** Describe intended use case, known limitations, and out-of-scope uses here.

- **Intended use:** _[describe]_  
- **Out-of-scope:** _[describe]_  
- **Known limitations:** _[describe]_  

---

*Generated by [SHAP Governance Framework](https://github.com/your-repo)*
"""
    return card


model_card_md = generate_model_card(
    model_name=MODEL_NAME,
    task_type=TASK_TYPE,
    feature_names=feature_names,
    shap_importance=importance_df[PRIMARY_MODEL],
    gov_results=gov_results[PRIMARY_MODEL],
    drift_df=drift_df,
    protected_attrs=PROTECTED_ATTRIBUTES,
    explainer_type_map=explainer_types
)

# Save to file
with open('MODEL_CARD.md', 'w') as f:
    f.write(model_card_md)

print("✅ Model card saved to MODEL_CARD.md\n")
display(Markdown(model_card_md))

---
## Section 8 — Production Deployment Checklist
### 8.1 Deployment Readiness Assessment

In [ ]:
def production_readiness_report(model_name, gov_results, additive_results, stability_df,
                                 drift_df, protected_attrs, explainer_type):
    """
    Comprehensive production deployment readiness assessment.
    """
    checks = []

    # 1. Governance gates
    gov = gov_results.get(model_name, {})
    gov_pass = all(c['pass'] for c in gov.values())
    checks.append(('Governance Quality Gates', gov_pass,
                   f"{sum(c['pass'] for c in gov.values())}/{len(gov)} checks passed"))

    # 2. Additive consistency
    add_res = next((r for r in additive_results if r['model'] == model_name), None)
    if add_res:
        checks.append(('SHAP Additive Consistency', add_res['passed'],
                       f"Max error: {add_res['max_err']:.6f}"))
    else:
        checks.append(('SHAP Additive Consistency', None, 'Approximate (KernelExplainer) — verify manually'))

    # 3. Attribution stability
    high_cv = stability_df[stability_df['cv'] > GOVERNANCE_CONFIG['max_cv_threshold']]
    checks.append(('Attribution Stability', len(high_cv) == 0,
                   f"{len(high_cv)} unstable feature(s)"))

    # 4. Drift baseline
    n_drifted = int(drift_df['Drifted'].sum())
    checks.append(('Drift Baseline Established', True,
                   f"Monitoring {len(drift_df)} features · {n_drifted} currently drifted"))

    # 5. Explainer type
    efficient = 'Kernel' not in explainer_type
    checks.append(('Optimised Explainer Selected', efficient,
                   f"{explainer_type} — {'fast & exact' if efficient else 'approximate, may impact latency'}"))

    # 6. Protected attributes
    checks.append(('Bias Audit Configured', bool(protected_attrs),
                   f"{len(protected_attrs)} attribute(s) monitored" if protected_attrs else 'Set PROTECTED_ATTRIBUTES to enable'))

    # 7. Model card
    card_exists = os.path.exists('MODEL_CARD.md')
    checks.append(('Model Card Generated', card_exists,
                   'MODEL_CARD.md written' if card_exists else 'Run Section 7 to generate'))

    # ── Print report ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  🚀  PRODUCTION READINESS REPORT — {model_name}")
    print(f"{'='*60}")

    passed = 0
    for item, status, note in checks:
        if status is True:
            icon, color = '✅', 'PASS'
            passed += 1
        elif status is False:
            icon, color = '❌', 'FAIL'
        else:
            icon, color = '⚠️ ', 'WARN'
            passed += 0.5

        print(f"  {icon}  {item:<38}  {note}")

    print(f"{'='*60}")
    score = int(passed / len(checks) * 100)
    verdict = '🟢 READY' if score >= 80 else ('🟡 CONDITIONAL' if score >= 60 else '🔴 NOT READY')
    print(f"  Score: {score}% | Verdict: {verdict}")
    print(f"{'='*60}\n")

    return score, checks


score, check_items = production_readiness_report(
    model_name=PRIMARY_MODEL,
    gov_results=gov_results,
    additive_results=additive_results,
    stability_df=stability_df,
    drift_df=drift_df,
    protected_attrs=PROTECTED_ATTRIBUTES,
    explainer_type=explainer_types[PRIMARY_MODEL]
)

### 8.2 Export Artefacts for Production

In [ ]:
import joblib

def export_production_artefacts(model_name, model, explainer, shap_importance,
                                 gov_results, output_dir='shap_artefacts'):
    """
    Export all production artefacts:
    - Trained model (.pkl)
    - SHAP explainer (.pkl)
    - Feature importance JSON
    - Governance report JSON
    """
    os.makedirs(output_dir, exist_ok=True)

    # Model
    model_path = f'{output_dir}/{model_name}_model.pkl'
    joblib.dump(model, model_path)
    print(f"✅ Model saved          → {model_path}")

    # Explainer
    exp_path = f'{output_dir}/{model_name}_explainer.pkl'
    joblib.dump(explainer, exp_path)
    print(f"✅ Explainer saved      → {exp_path}")

    # Feature importance
    imp_path = f'{output_dir}/{model_name}_feature_importance.json'
    imp_dict = shap_importance.to_dict()
    with open(imp_path, 'w') as f:
        json.dump({str(k): float(v) for k, v in imp_dict.items()}, f, indent=2)
    print(f"✅ Feature importance   → {imp_path}")

    # Governance report
    gov_path = f'{output_dir}/{model_name}_governance.json'
    gov_clean = {
        k: {kk: (bool(vv) if isinstance(vv, (bool, np.bool_)) else vv)
            for kk, vv in v.items()}
        for k, v in gov_results.items()
    }
    report = {
        'model': model_name,
        'generated': datetime.now().isoformat(),
        'explainer': explainer_types[model_name],
        'governance': gov_clean
    }
    with open(gov_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"✅ Governance report    → {gov_path}")

    print(f"\n📦 All artefacts exported to ./{output_dir}/")


export_production_artefacts(
    model_name=PRIMARY_MODEL,
    model=models[PRIMARY_MODEL],
    explainer=explainers[PRIMARY_MODEL],
    shap_importance=importance_df[PRIMARY_MODEL],
    gov_results=gov_results[PRIMARY_MODEL]
)

### 8.3 Inference Helper — Real-Time Explanation

In [ ]:
def explain_single_prediction(model, explainer, input_data: pd.DataFrame,
                               feature_names: list, task_type: str, top_n: int = 5):
    """
    Production-ready function to explain a single prediction.

    Parameters
    ----------
    model        : trained sklearn-compatible model
    explainer    : fitted SHAP explainer
    input_data   : pd.DataFrame with a single row (or batch)
    feature_names: list of feature column names
    task_type    : 'classification' or 'regression'
    top_n        : number of top features to return

    Returns
    -------
    dict with prediction, confidence, top_features, shap_values, base_value
    """
    sv = explainer(input_data)
    vals = sv.values[0]
    if vals.ndim == 2:
        vals = vals[:, 1]    # binary: class 1
    base = float(sv.base_values[0, 1] if sv.base_values.ndim > 1 else sv.base_values[0])

    if task_type == 'classification' and hasattr(model, 'predict_proba'):
        pred = model.predict(input_data)[0]
        prob = model.predict_proba(input_data)[0].max()
    else:
        pred = model.predict(input_data)[0]
        prob = None

    top_idx = np.argsort(np.abs(vals))[::-1][:top_n]
    top_features = [
        {'feature': feature_names[i], 'shap_value': round(float(vals[i]), 5),
         'feature_value': float(input_data.iloc[0, i])}
        for i in top_idx
    ]

    return {
        'prediction': pred,
        'confidence': round(float(prob), 4) if prob is not None else None,
        'base_value': round(base, 5),
        'top_features': top_features,
        'all_shap_values': dict(zip(feature_names, vals.round(5).tolist()))
    }


# ── Demo call ─────────────────────────────────────────────────────────────────
X_eval_primary = X_test_scaled if PRIMARY_MODEL in ('Linear', 'NeuralNetwork') else X_test
sample_row = X_eval_primary.iloc[[0]]

explanation = explain_single_prediction(
    model=models[PRIMARY_MODEL],
    explainer=explainers[PRIMARY_MODEL],
    input_data=sample_row,
    feature_names=feature_names,
    task_type=TASK_TYPE
)

print("🔍 Real-time explanation output (JSON):")
print(json.dumps({k: v for k, v in explanation.items() if k != 'all_shap_values'}, indent=2))

---
## ✅ Framework Complete

### Summary of Outputs

| Artefact | Description |
|----------|-------------|
| `MODEL_CARD.md` | Auto-generated explainability model card |
| `shap_global_importance_comparison.png` | Global SHAP importance across all models |
| `shap_dependence_plots.png` | Top-3 feature dependence plots |
| `shap_heatmap.png` | Attribution heatmap across samples |
| `shap_drift_monitor.png` | SHAP drift visualisation |
| `shap_artefacts/` | Model, explainer, importance & governance JSONs |

---

### Next Steps for GitHub

1. **Add your data**: Replace the demo data block in Section 0.3 with your own dataset
2. **Set protected attributes**: Update `PROTECTED_ATTRIBUTES` for full bias auditing
3. **Configure thresholds**: Adjust `GOVERNANCE_CONFIG` to match your team's standards
4. **Extend the model registry**: Add your production model in Section 1.1
5. **Schedule drift checks**: Integrate Section 6 into your ML monitoring pipeline

---

### References

- [SHAP Documentation](https://shap.readthedocs.io/en/latest/)
- [Lundberg & Lee (2017) — A Unified Approach to Interpreting Model Predictions](https://arxiv.org/abs/1705.07874)
- [EU AI Act — Explainability Requirements](https://artificialintelligenceact.eu/)
- [NIST AI Risk Management Framework](https://www.nist.gov/system/files/documents/2023/01/26/AI_RMF_1.0.pdf)

---
*SHAP Governance Framework · MIT License*